In [4]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv


In [5]:
load_dotenv()
model = ChatGroq(model = 'openai/gpt-oss-120b')

In [6]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [7]:
def create_outline(state: BlogState) -> BlogState:
    # fetch title
    title = state['title']

    # call llm to generate outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [8]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content
    state['content'] = content

    return state

In [9]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [13]:
initial_state = {'title': 'Rise of AI in India'}
final_state = workflow.invoke(initial_state)

print(final_state['outline'])

**Blog Title:** *The Rise of AI in India: Opportunities, Challenges, and the Road Ahead*  

---

## 1. Introduction  
   - **Hook:** A striking statistic or anecdote (e.g., “India’s AI market is projected to hit $17 billion by 2027”)  
   - **Why It Matters:** The strategic importance of AI for India’s economy, society, and global standing.  
   - **Scope of the Post:** What readers will learn – history, current landscape, key players, sectoral impact, policy, challenges, and future outlook.  

---

## 2. Historical Context: From Early Experiments to a National Priority  
   1. **Early Foundations (1990s‑2000s)**  
      - Academic research in IITs, ISI, and IISc.  
      - First Indian AI startups (e.g., Nucleus Software, iLabs).  
   2. **Policy Shift (2015‑2020)**  
      - NITI Aayog’s “Artificial Intelligence #AIforAll” report (2018).  
      - Launch of the **National AI Strategy** and the **Digital India** programme.  
   3. **Catalysts for Growth**  
      - Availability of che